# Voces con Qwen3-TTS · vídeos CONAF

Genera la narración de un vídeo del pipeline `coipo_notebooklm/video` con
**Qwen3-TTS CustomVoice (0,6 B)**: nueve voces preconstruidas, sin clonar a
nadie. Se eligen dos de oído en la celda 4 — la que suene menos extranjera
hablando español.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.

Al final descarga `narracion_<nombre>.mp3` y `beats_<nombre>.json`. **Van
siempre juntos**: el JSON lleva la duración exacta de cada frase y es lo único
que sostiene la sincronía del vídeo.

**El orden importa:** la celda 4 es un casting corto y la 4b el intercambio de
prueba. No pases a la 5 hasta que las voces te gusten — las celdas 5 a 7 tardan
y sería tirarlas.

> **Sin probar.** La máquina donde se escribió no tiene GPU. Está hecho contra
> la API publicada de Qwen3-TTS. Las celdas 4 y 4b son cortas a propósito, para
> que un fallo salte en un minuto y no en veinte.


## 1 · Comprobar la GPU

In [ ]:
import subprocess, torch

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), (
    'Sin GPU. Entorno de ejecucion -> Cambiar tipo de entorno -> GPU')
cap = torch.cuda.get_device_capability()
print('capacidad CUDA:', cap)

# flash-attention-2 exige Ampere (8.0+). La T4 gratuita de Colab es Turing (7.5)
# y revienta al cargar el modelo si se le pide. Se vuelve a calcular en la
# celda 3, porque el reinicio de la celda 2 borra todo lo de aqui.
print('attn_implementation sera:',
      'flash_attention_2' if cap[0] >= 8 else 'sdpa')

## 2 · Instalar · tarda ~2 minutos

`qwen-tts` fija `transformers==4.57.3` exacto, y Colab trae preinstalada una v5
donde `GenerationMixin` ya no vive en `transformers.generation`. Pip instala la
correcta, pero **el kernel sigue con la vieja cargada en memoria**, así que la
celda 3 fallaría con un `ImportError` críptico si no se reinicia.

**Cuando esta celda termine, reinicia tú:** *Entorno de ejecución → Reiniciar
sesión*. Luego sigue **desde la celda 3**, sin repetir ésta.

> Antes esta celda se reiniciaba sola, y era mala idea: en los logs deja la
> misma firma que una caída de verdad —`restarting kernel (1/5)` y un `atexit`
> roto— así que era imposible saber si había ido bien. Reiniciando a mano no hay
> ambigüedad.

In [ ]:
# Versiones EXACTAS, comprobadas en PyPI, no dejadas a la resolucion de pip.
# Colab trae transformers v5 y huggingface-hub 1.x preinstalados, y de ahi sale
# el choque; fijandolo todo, la instalacion es determinista.
!pip -q install qwen-tts==0.1.1 transformers==4.57.3 huggingface-hub==0.36.2 tokenizers==0.22.2 accelerate==1.12.0 soundfile
# sox NO es solo el paquete de pip: qwen-tts llama al BINARIO, y Colab no lo
# trae. Sin esto sale «/bin/sh: 1: sox: not found» al importar.
!apt-get -qq install -y ffmpeg sox libsox-fmt-all > /dev/null
!which sox || echo 'OJO: sox sigue sin instalarse'

ESPERADO = {'qwen-tts': '0.1.1', 'transformers': '4.57.3', 'huggingface-hub': '0.36.2', 'tokenizers': '0.22.2', 'accelerate': '1.12.0'}

import importlib.metadata as md
malas = []
for pkg, quiero in ESPERADO.items():
    try:
        hay = md.version(pkg)
    except Exception:
        hay = 'NO INSTALADO'
    if hay != quiero:
        malas.append(pkg)
    print('%-18s %-12s %s' % (pkg, hay,
                              'ok' if hay == quiero else 'DISTINTA, esperaba ' + quiero))
print()
if malas:
    print('OJO, no cuadran:', ', '.join(malas))

# El aviso de pip sobre diffusers es inofensivo: pide huggingface-hub 1.x, pero
# no lo usamos y aqui manda transformers.
#
# NO se reinicia solo a proposito. Un reinicio automatico deja en los logs la
# misma firma que una caida de verdad —«restarting kernel (1/5)» y un atexit
# roto—, y entonces es imposible saber si fue intencionado o si el proceso se
# murio. Reiniciando a mano no hay ambiguedad.
print()
print('=' * 62)
print('  INSTALACION TERMINADA.')
print()
print('  Ahora, A MANO:  Entorno de ejecucion -> Reiniciar sesion')
print('  Y despues sigue DESDE LA CELDA 3. No repitas esta.')
print('=' * 62)

## 3 · Cargar el modelo

El **0,6 B** en lugar del 1,7 B: el grande tumbaba el kernel en la Colab
gratuita, que tiene 12,7 GB de RAM de sistema y ya lleva TensorFlow cargado.

El precio de bajar: `VoiceDesign` —describir la voz con texto— **sólo existe en
1,7 B**. Con `CustomVoice` hay que elegir entre nueve voces fijas.

In [ ]:
import torch, importlib.metadata as md

# Comprobar ANTES de importar: si las versiones no son las que qwen-tts fija, el
# fallo sale aqui y dice que hacer, en vez de como un ImportError sobre
# GenerationMixin tres marcos mas abajo.
ESPERADO = {'qwen-tts': '0.1.1', 'transformers': '4.57.3', 'huggingface-hub': '0.36.2', 'tokenizers': '0.22.2', 'accelerate': '1.12.0'}
hay = {}
for k in ESPERADO:
    try:
        hay[k] = md.version(k)
    except Exception:
        hay[k] = 'NO INSTALADO'
print(' | '.join('%s %s' % (k, v) for k, v in hay.items()))

malas = [k for k in ESPERADO if hay[k] != ESPERADO[k]]
if malas:
    raise SystemExit('Versiones equivocadas en: ' + ', '.join(malas) +
                     '. Ejecuta la celda 2 y REINICIA el entorno antes de volver aqui.')

cap = torch.cuda.get_device_capability()
ATTN = 'flash_attention_2' if cap[0] >= 8 else 'sdpa'


def memoria(cuando):
    import psutil
    ram = psutil.virtual_memory()
    vram = torch.cuda.memory_allocated() / 2**30 if torch.cuda.is_available() else 0
    print('%-22s RAM %.1f/%.1f GB libres %.1f | VRAM usada %.1f GB'
          % (cuando, (ram.total - ram.available) / 2**30, ram.total / 2**30,
             ram.available / 2**30, vram))


# La RAM del sistema es la que mata el kernel en Colab, no la VRAM: el proceso
# se muere sin traza y solo se ve «restarting kernel (1/5)» en los logs. Por eso
# se mide antes y despues.
memoria('antes de importar')
from qwen_tts import Qwen3TTSModel
memoria('tras importar')

MODELO = 'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice'
# 0.6B CustomVoice en vez de 1.7B VoiceDesign: el 1.7B tumbaba el kernel en la
# Colab gratuita, que solo tiene 12,7 GB de RAM de sistema y ya lleva
# TensorFlow cargado. El precio es que la voz ya no se describe con texto:
# CustomVoice trae nueve voces fijas y hay que elegir dos de oido (celda 4).
# El `instruct` sigue existiendo y sirve para el tono y el enfasis.
try:
    modelo = Qwen3TTSModel.from_pretrained(
        MODELO, device_map='cuda:0', dtype=torch.bfloat16,
        attn_implementation=ATTN, low_cpu_mem_usage=True)
except Exception as e:
    print('FALLO al cargar:', type(e).__name__, e)
    memoria('en el fallo')
    raise

memoria('modelo cargado')
print('listo:', MODELO, 'con', ATTN)

## 4 · Casting · **escucha las nueve y elige dos**

`CustomVoice` no deja describir la voz con texto: trae **nueve voces fijas**, con
nombres que no dicen nada sobre cómo suenan en español. Cuál se acerca más al
chileno **sólo se sabe oyéndolas**.

Esta celda hace decir a las nueve la misma frase en español. Escúchalas y anota
**una femenina y una masculina** — las que suenen menos extranjeras.

Cada una va precedida de su nombre dicho en voz alta, para no perderse.

In [ ]:
import soundfile as sf, numpy as np, IPython.display as ipd

# Los nueve nombres publicados de CustomVoice. Si alguno no existiera en esta
# version del modelo, se salta con aviso en vez de tumbar la celda.
CANDIDATAS = ['Vivian', 'Serena', 'Ono_Anna', 'Sohee',
              'Dylan', 'Eric', 'Ryan', 'Aiden', 'Uncle_Fu']

FRASE = ('Y bosques, dieciocho coma nueve millones de hectareas. '
         'Un veinticinco por ciento del pais.')

piezas, sr, sirven = [], 24000, []
for nombre in CANDIDATAS:
    try:
        # Primero el nombre, con la MISMA voz, para saber cual es cual al oirlo.
        w, sr = modelo.generate_custom_voice(text=nombre.replace('_', ' '),
                                             language='Spanish', speaker=nombre)
        piezas.append(np.asarray(w[0] if np.ndim(w) > 1 else w, dtype=np.float32))
        piezas.append(np.zeros(int(0.25 * sr), dtype=np.float32))

        w, sr = modelo.generate_custom_voice(
            text=FRASE, language='Spanish', speaker=nombre,
            instruct='Tono conversacional de podcast, natural y cercano.')
        piezas.append(np.asarray(w[0] if np.ndim(w) > 1 else w, dtype=np.float32))
        piezas.append(np.zeros(int(0.60 * sr), dtype=np.float32))
        sirven.append(nombre)
        print('ok  ', nombre)
    except Exception as e:
        print('FALLA', nombre, '->', type(e).__name__, e)

print()
print('voces utilizables:', ', '.join(sirven))
audio = np.concatenate(piezas)
sf.write('casting.wav', audio, sr)
print('%.0f s de casting' % (audio.size / sr))
ipd.Audio('casting.wav')

### 4b · Anota tus dos elegidas

Pon aquí los nombres y vuelve a escuchar el intercambio completo, ya con las
mismas frases de la muestra C que oíste con edge-tts. **Compáralo con
`video/muestra-voces-C-chilenas-con-coqueteo.mp3` del repo**: si Qwen no gana
claramente, no vale la pena el cambio.

**Sobre el tono entre las dos voces:** una cercanía cómplice apenas perceptible,
que se escucha en cómo se escuchan y no en lo que dicen. Techo duro: **nunca
explícito**, nunca verbalizado, nunca seductor. Profesional y cálido siempre.

In [ ]:
VOZ_C = 'Serena'      # <- la femenina que elegiste
VOZ_L = 'Eric'        # <- la masculina que elegiste
VOCES = {'c': VOZ_C, 'l': VOZ_L}

# El coqueteo va en la ENTREGA, no en las palabras.
CERCANIA = (' Se dirige a su companero de programa con una cercania complice '
            'apenas perceptible: escucha de verdad y se le nota que disfruta la '
            'conversacion. Nunca coqueto de forma evidente ni seductor; '
            'profesional y calido en todo momento.')

PRUEBA = [
    ('c', 'Y bosques, dieciocho coma nueve millones. Un veinticinco por ciento del pais.',
     ' Enfasis claro en la cifra: mas lenta, mas alta y mas fuerte.'),
    ('l', 'Veinticinco por ciento? Eso suena a titular.', ' Reaccion rapida, con chispa.'),
    ('c', 'Suena. Y ahi esta justo la trampa.', ' Reaccion rapida, con chispa.'),
    ('l', 'Ya decia yo que te traias algo entre manos.', ' Mas bajo y calido, con complicidad.'),
    ('c', 'Me conoces demasiado bien para lo poco que llevamos.',
     ' Mas bajo y calido, con complicidad.'),
]

piezas = []
for quien, texto, matiz in PRUEBA:
    w, sr = modelo.generate_custom_voice(
        text=texto, language='Spanish', speaker=VOCES[quien],
        instruct='Tono conversacional de podcast.' + CERCANIA + matiz)
    piezas.append(np.asarray(w[0] if np.ndim(w) > 1 else w, dtype=np.float32))
    piezas.append(np.zeros(int(0.30 * sr), dtype=np.float32))

audio = np.concatenate(piezas)
sf.write('prueba.wav', audio, sr)
print('%s y %s | %.1f s' % (VOZ_C, VOZ_L, audio.size / sr))
ipd.Audio('prueba.wav')

## 5 · Traer el guion desde GitHub

Se descarga la versión viva del repo, para no copiar y pegar texto — que es
justo como se desincronizan las cosas.

In [ ]:
NOMBRE = 'catastro'      # catastro | contrato_pod | ecosistema_pod
RAMA = 'videos-v2'
BASE = ('https://raw.githubusercontent.com/Sud-Austral/coipo_notebooklm/'
        + RAMA + '/video/pipeline')

import urllib.request, importlib.util
urllib.request.urlretrieve(BASE + '/guion_' + NOMBRE + '.py', 'guion.py')
spec = importlib.util.spec_from_file_location('g', 'guion.py')
g = importlib.util.module_from_spec(spec)
spec.loader.exec_module(g)
GUION = g.GUION

print(len(GUION), 'latidos |', sum(1 for b in GUION if '*' in b['t']), 'con enfasis')
print('primero:', GUION[0]['t'])

## 6 · Sintetizar

Dos diferencias con el pipeline local, y las dos salen de cómo funciona Qwen:

- **No hay parámetro de velocidad.** El énfasis se pide en lenguaje natural por
  el `instruct`, y la aceleración global se aplica después con `atempo`, que
  **conserva el tono**. Reescalar la muestra subiría el pitch y dejaría las
  voces agudas.
- **Se recorta el silencio de cada trozo**, por la misma razón medida en local:
  sin eso, una pausa de 0,15 s del guion acaba durando 0,75 s y la conversación
  se arrastra.

La barra cuenta **trozos**, no latidos: un latido con énfasis son dos o tres
llamadas al modelo, así que contar latidos daría un tiempo restante mentiroso.

In [ ]:
import json, time, subprocess
from tqdm.auto import tqdm

ACELERACION = 1.10     # 10 % mas rapido, como la muestra C elegida
COSTURA = 0.055        # micro-pausa entre trozos de una misma frase
PORTADILLA = 10.0      # los 10 s de Forestin, intocables
HZ = 24000

MATIZ = {'normal': 'tono conversacional, natural.',
         'lento': 'mas pausado y grave, subrayando la idea.',
         'vivo': 'reaccion rapida, con chispa.',
         'suave': 'mas bajo y calido, con complicidad.'}


def trozos(t):
    crudos = [(x, i % 2 == 1) for i, x in enumerate(t.split('*')) if x]
    fuera = []
    for x, mk in (crudos or [(t, False)]):
        if not any(c.isalnum() for c in x) and fuera:
            fuera[-1] = (fuera[-1][0] + x, fuera[-1][1])
        else:
            fuera.append((x, mk))
    return fuera


def recortar(a, guarda=0.09, umbral=0.006):
    if a.size == 0:
        return a
    pico = float(np.abs(a).max()) or 1.0
    idx = np.nonzero(np.abs(a) > pico * umbral)[0]
    if idx.size == 0:
        return a
    g = int(guarda * HZ)
    return a[max(0, idx[0] - g):min(a.size, idx[-1] + g)]


def decir(texto, hablante, matiz, enfatico):
    # En CustomVoice el TIMBRE lo fija `speaker` y el `instruct` solo maneja
    # tono y enfasis. En VoiceDesign el instruct hacia las dos cosas.
    ins = 'Tono conversacional de podcast.' + CERCANIA + ' ' + matiz
    if enfatico:
        ins += ' Marca esta parte con enfasis claro: mas lenta, mas alta y mas fuerte.'
    w, sr = modelo.generate_custom_voice(text=texto, language='Spanish',
                                         speaker=hablante, instruct=ins)
    a = np.asarray(w[0] if np.ndim(w) > 1 else w, dtype=np.float32)
    if sr != HZ:
        n = int(round(a.size * HZ / sr))
        a = np.interp(np.linspace(0, a.size - 1, n), np.arange(a.size), a)
        a = a.astype(np.float32)
    return a


plan = [(i, b, trozos(b['t'])) for i, b in enumerate(GUION)]
barra = tqdm(total=sum(len(x[2]) for x in plan), desc='sintetizando', unit='trozo')

piezas, meta, t = [], [], PORTADILLA
t_ini = time.time()

for i, b, texto_trozos in plan:
    partes = []
    for j, (txt, mk) in enumerate(texto_trozos):
        if j:
            partes.append(np.zeros(int(COSTURA * HZ), dtype=np.float32))
        partes.append(recortar(decir(txt.strip(), VOCES[b['v']],
                                     MATIZ[b.get('tono', 'normal')], mk)))
        barra.update(1)
    audio = np.concatenate(partes)
    pausa = float(b.get('p', 0.3))
    piezas.append(audio)
    piezas.append(np.zeros(int(pausa * HZ), dtype=np.float32))
    # Las duraciones del JSON son las de DESPUES de acelerar.
    meta.append(dict(i=i, v=b['v'], inicio=round(t, 3),
                     dur=round(audio.size / HZ / ACELERACION, 3),
                     foto=b['foto'], z=b.get('z', 'completo'),
                     rot=b.get('rot'), tono=b.get('tono', 'normal'),
                     texto=b['t'].replace('*', '')))
    t += (audio.size / HZ + pausa) / ACELERACION
    barra.set_postfix_str('%d/%d latidos - %.0f s de audio' % (i + 1, len(GUION), t))

barra.close()
sf.write('crudo.wav', np.concatenate(piezas), HZ)
print('sintesis terminada en %.1f min' % ((time.time() - t_ini) / 60))

## 7 · Acelerar, normalizar y comprobar

In [ ]:
mp3 = 'narracion_' + NOMBRE + '.mp3'
subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', 'crudo.wav',
                '-af', 'atempo=' + str(ACELERACION) + ',loudnorm=I=-18:TP=-2:LRA=11',
                '-b:a', '160k', mp3], check=True)

fichero = 'beats_' + NOMBRE + '.json'
with open(fichero, 'w', encoding='utf-8') as f:
    json.dump(dict(portadilla=PORTADILLA, total=round(t, 3), beats=meta),
              f, ensure_ascii=False, indent=1)

d = float(subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                          '-of', 'csv=p=0', mp3], capture_output=True, text=True).stdout)
print('%s  %.1f s  (%d min %02d s)' % (mp3, d, d // 60, d % 60))
print('el JSON dice %.1f s  ->  desfase %.2f s' % (t, abs(d - t)))
if abs(d - t) > 1.0:
    print('DESFASE ALTO: el video saldria descuadrado. No uses estos archivos.')
ipd.Audio(mp3)

## 8 · Descargar

`narracion_*.mp3` va a `video/public/`, y `beats_*.json` a `video/src/` **y** a
`video/pipeline/`. Después, en el PC:

```
cd video && npx remotion render src/index.jsx Catastro out/catastro.mp4
```

In [ ]:
from google.colab import files

files.download(mp3)
files.download(fichero)